<a href="https://colab.research.google.com/github/Masaki-Sk/100-knock/blob/main/ja/ch04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [ ]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

In [ ]:
!pip install janome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 94.1 MB/s eta 0:00:00


In [ ]:
from janome.tokenizer import Tokenizer

text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

t = Tokenizer()

verbs = []

for token in t.tokenize(text):
    pos = token.part_of_speech.split(",")[0]

    if pos == "動詞":
        verbs.append(token.surface)

print(verbs)

['し', '除か', 'なら', 'し', 'わから', '吹き', '遊ん', '暮し', '来']


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

In [ ]:
verbs_base = []

for token in t.tokenize(text):
    pos = token.part_of_speech.split(",")[0]

    if pos == "動詞":
        verbs_base.append(token.base_form)

print(verbs_base)

['する', '除く', 'なる', 'する', 'わかる', '吹く', '遊ぶ', '暮す', '来る']


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。

In [ ]:
noun_no_noun_phrases = []

tokens = list(t.tokenize(text))

for i in range(len(tokens) - 2):
    t0 = tokens[i]
    t1 = tokens[i + 1]
    t2 = tokens[i + 2]

    pos0 = t0.part_of_speech.split(",")[0]
    pos1 = t1.part_of_speech.split(",")[0]
    pos2 = t2.part_of_speech.split(",")[0]

    if pos0 == "名詞" and t1.surface == "の" and pos2 == "名詞":
        phrase = t0.surface + t1.surface + t2.surface
        noun_no_noun_phrases.append(phrase)

print(noun_no_noun_phrases)

['暴虐の王', '村の牧人']


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

In [ ]:
!pip install ja-ginza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.9 MB/s eta 0:00:00


In [ ]:
import spacy

nlp = spacy.load("ja_ginza")

doc = nlp(text)

pairs = []

for token in doc:
    if token.head == token:
        continue

    pairs.append((token.text, token.head.text))

for dep, head in pairs:
    print(f"{dep}\t{head}")


	メロス
メロス	激怒
は	メロス
し	激怒
た	激怒
。	激怒
必ず	除か
、	必ず
かの	暴虐
邪智	暴虐
暴虐	王
の	暴虐
王	除か
を	王
除か	決意
なけれ	除か
ば	なけれ
なら	なけれ
ぬ	なけれ
と	除か
し	決意
た	決意
。	決意

	メロス
メロス	わから
に	メロス
は	メロス
政治	わから
が	政治
ぬ	わから
。	わから

	メロス
メロス	牧人
は	メロス
、	メロス
村	牧人
の	村
で	牧人
ある	で
。	牧人

	笛
笛	吹き
を	笛
吹き	暮し
、	吹き
羊	遊ん
と	羊
遊ん	暮し
で	遊ん
て	暮し
来	て
た	暮し
。	暮し

	邪悪
けれど	

も	

邪悪	敏感
に	邪悪
対し	に
ては	に
、	邪悪
人	倍
一	倍
倍	敏感
に	倍
で	敏感
あっ	で
た	敏感
。	敏感


## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

In [ ]:
def find_verb(node, tokens):
  tokens.append(node)

  if node.pos_ == "VERB":
    return tokens

  for child in node.children:
    if child.dep_ in ("aux", "cop", "fixed"):
      v = find_verb(child, tokens)

      if v and v[-1] is not None and v[-1].pos_ == "VERB":
        return v

  return tokens

def attach_child(parent, tokens):

  for child in parent.children:
    if child.dep_ in ("aux", "cop"):
      tokens.append(child)
      attach_child(child, tokens)

  return tokens

predicates = []

for token in doc:
  if token.text == "メロス" and token.dep_.startswith("nsubj"):
    path = find_verb(token.head, [])
    head = path[-1]

    pred_tokens = attach_child(head, path)

    pred_text = "".join(tok.text for tok in pred_tokens)

    predicates.append(pred_text)

print(predicates)

['激怒した', '牧人である']


## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

In [ ]:
def print_tree(token, children, indent=0):
    print("-" * indent + token.text)
    for child in children.get(token.i, []):
        if child is token:
            continue
        print_tree(child, children, indent + 1)

text_clip = "メロスは激怒した。"
doc_clip = nlp(text_clip)

children = {}
root = None

for token in doc_clip:
    if token.head == token:
        root = token
    children.setdefault(token.head.i, []).append(token)

print_tree(root, children)

激怒
-メロス
--は
-し
-た
-。


## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。